# Datenreduktion im hierarchischen Ansatz

**Autor:** jd · **Datum:** 2026-10-20

**Ziel:** Die Datenreduktionsstudie (#155) für den hierarchischen Ansatz wiederholen und um das ergänzen, was nur hierarchisch geht (#163, #162): Jede Ebene wählt ihre eigenen Bänder. Das Crop-Modell wählt Bänder für die Kultur, jedes Stage-Modell Bänder für die Stadien seiner Kultur.

**Hypothesen:**

1. Sentinel-2 reicht auch hierarchisch (wie bei der kombinierten SVM).
2. Eine eigene Bandauswahl je Ebene schlägt die globale Auswahl, die im kombinierten Ansatz schwach war (0,739 bei 10 Bändern).

**Vorgehen:** 80/20-Split, CV auf dem Trainingsteil, Experiment `band-reduction`, approach `hierarchical`. Reihenfolge wie in `08_datenreduktion/02_jd_datenreduktion`: Feature-Audit, Trade-off-Kurve, Bandauswahl je Ebene, Empfehlung nach CV-Regel, **eine** Validierung.

In [ ]:
%load_ext autoreload
%autoreload 2

from typing import Any

import matplotlib.pyplot as plt
import mlflow
import pandas as pd
from sklearn.pipeline import Pipeline

from awp2.config import (
    MLFLOW_REDUCTION_EXPERIMENT,
    MLFLOW_TRACKING_URI,
    MODEL_DOCS_FIGURES_DIR,
    N_MEASURED_BANDS_COL,
    REDUCTION_BEST_PARAMS_COL,
    REDUCTION_FAMILY_COL,
    REDUCTION_MAX_CV_LOSS,
    REDUCTION_TUNING_RUN_COL,
    REDUCTION_VARIANT_COL,
)
from awp2.data import load_split
from awp2.experiment import RunConfig, TuneConfig, run, tune
from awp2.models import make_hierarchical_reduced_svm, make_hierarchical_svm, oracle_crop_metrics
from awp2.plots import plot_metric_comparison, plot_tradeoff, save_doc_figure
from awp2.preprocessing import PreprocessingConfig, build_preprocessor
from awp2.reduction import CURVE_METRICS, ReductionVariant, reduction_curve, reduction_variants

## 1. Referenz und Suchraum

Referenz ist die hierarchische RBF-SVM (bisher bestes Setup: Kultur `C=30`, Stadium `C=100`). Pro Ebene wird nur `gamma` gesucht, fest oder `"scale"`. `"scale"` passt sich der Merkmalszahl der reduzierten Eingaben an.

In [ ]:
APPROACH = "hierarchical"
model = make_hierarchical_svm()
base = PreprocessingConfig(use_spectral_standard_scale=True)
grid: dict[str, list[Any]] = {
    "crop_estimator__C": [30.0],
    "crop_estimator__gamma": [0.01, "scale"],
    "stage_estimator__C": [100.0],
    "stage_estimator__gamma": [0.003, "scale"],
}
variants = reduction_variants(base)


def template(name: str, preprocessing: PreprocessingConfig = base) -> TuneConfig:
    return TuneConfig(
        name=name,
        approach=APPROACH,
        experiment=MLFLOW_REDUCTION_EXPERIMENT,
        description=f"Datenreduktion hierarchisch (#163): {name}",
        param_grid=grid,
        preprocessing=preprocessing,
    )

## 2. Feature-Audit

Wie in #155: je eine Merkmalsgruppe weglassen oder ergänzen.

In [ ]:
audit_configs = {
    "Spektren + Meta (Referenz)": base,
    "Nur Spektren": base.model_copy(update={"use_meta": False}),
    "+ Vegetationsindizes": base.model_copy(update={"use_vegetation_indices": True}),
    "+ Mittlere Albedo": base.model_copy(update={"use_mean_albedo": True}),
    "+ PC_1/PC_2": base.model_copy(update={"use_principal_components": True}),
    "Nur Indizes + Meta": PreprocessingConfig(
        use_spectral_standard_scale=True, use_spectra=False, use_vegetation_indices=True
    ),
    "Nur Meta": PreprocessingConfig(use_spectra=False),
}
audit_slugs = ["reference", "no_meta", "indices", "albedo", "pcs", "indices_meta", "meta"]
audit_rows = []
for (label, preprocessing), slug in zip(audit_configs.items(), audit_slugs, strict=True):
    tuned = tune(model, template(f"hier_audit_{slug}", preprocessing))
    best = tuned.candidates.loc[tuned.candidates["rank"].idxmin()]
    audit_rows.append(
        {
            "Merkmale": label,
            "CV-BAcc kombiniert": tuned.best_score,
            "Std": best["std"],
            **tuned.input_size._asdict(),
        }
    )
audit = pd.DataFrame(audit_rows).set_index("Merkmale")
audit.round(3)

**Befund Feature-Audit:** dasselbe Muster wie im kombinierten Ansatz (#155), alle Werte etwa 0,01–0,05 niedriger.

- AEZ und Monat sind unverzichtbar: ohne sie 0,739 statt 0,846, allein 0,486.
- Indizes (0,839), Albedo (0,846) und PC_1/PC_2 (0,832) bringen nichts.
- Nur Indizes + Meta erreicht 0,821 mit 28 Bändern.

## 3. Trade-off-Kurve (globale Reduktion)

Dieselben 17 Eingaben wie in #155, jetzt mit der hierarchischen SVM. Zum Vergleich werden die kombinierten CV-Werte aus MLflow gelesen (gleicher Split, Experiment `band-reduction`).

In [ ]:
curve = reduction_curve(model, variants, template("hier_reduction"))
curve.drop(columns=[REDUCTION_BEST_PARAMS_COL, REDUCTION_TUNING_RUN_COL]).round(3)

In [ ]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
combined_runs = mlflow.search_runs(
    experiment_names=[MLFLOW_REDUCTION_EXPERIMENT],
    filter_string=(
        "params.`split.validation_size` = '0.2' and tags.`run.kind` = 'tuning' "
        "and attributes.run_name LIKE 'reduction_svm_%'"
    ),
)
combined_cv = combined_runs.assign(
    variant=combined_runs["tags.mlflow.runName"].str.removeprefix("reduction_svm_")
).set_index("variant")["metrics.cv_bacc_combined_mean"]
comparison = curve.set_index(REDUCTION_VARIANT_COL)[[N_MEASURED_BANDS_COL, "bacc_combined"]].rename(
    columns={"bacc_combined": "hierarchisch"}
)
comparison["kombiniert"] = combined_cv
comparison["Differenz"] = comparison["hierarchisch"] - comparison["kombiniert"]
comparison.round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
for ax, metric in zip(axes, CURVE_METRICS, strict=True):
    plot_tradeoff(curve, metric=metric, ax=ax)
    ax.set_title(ax.get_ylabel())
fig.suptitle("Hierarchisch: Qualität vs. gemessene Bänder")
fig.tight_layout()
save_doc_figure(fig, "hierarchisch_reduktion_tradeoff", MODEL_DOCS_FIGURES_DIR)

**Befund Trade-off:**

- **Hierarchisch liegt bei jeder Eingabe unter dem kombinierten Ansatz:** mit allen Bändern 0,846 gegenüber 0,859 (−0,012). Am stärksten verliert er bei sehr wenig Information: PCA-2 −0,083, Bandauswahl −0,03 bis −0,05, nur Meta −0,051.
- **Die Form der Kurve ist dieselbe:**
    - PCA ab 20 Komponenten und jedes 2./4./8. Band halten die Qualität (0,839–0,846).
    - Die univariate Bandauswahl fällt deutlich ab (0,694 bei 10 Bändern).
- **Sentinel-2 verliert hierarchisch etwas mehr** (0,834, −0,012 gegenüber allen Bändern; kombiniert ±0). Der Verlust liegt beim Stadium (0,881 statt 0,902), die Kultur hält (0,930 statt 0,938). Die Stage-Modelle pro Kultur sehen nur die Zeilen ihrer Kultur, und mit 10 breiten Bändern fehlen ihnen offenbar feine Unterschiede, die die gemeinsame SVM über alle Kulturen hinweg ausgleicht.

## 4. Bandauswahl je Ebene (#162)

Statt einer globalen Auswahl wählt jede Ebene ihre `k` Bänder selbst. Gesucht wird `k` getrennt für Kultur und Stadium. Gemessen werden muss danach die **Vereinigung** aller gewählten Bänder: Für jede Zeile ist vorher nicht bekannt, welches Stage-Modell sie braucht. Diese Vereinigung steht in der Spalte `n_measured_bands`.

In [ ]:
per_level_rows = []
for k in (5, 10, 20, 40):
    per_level = reduction_curve(
        make_hierarchical_reduced_svm(k),
        [ReductionVariant(name=f"per_level_{k}", family="band_selection", preprocessing=base)],
        TuneConfig(
            name=f"hier_per_level_{k}",
            approach=APPROACH,
            experiment=MLFLOW_REDUCTION_EXPERIMENT,
            description=f"Bandauswahl je Ebene (#162), k={k}",
            param_grid={
                "crop_estimator__classifier__C": [30.0],
                "crop_estimator__classifier__gamma": [0.01, "scale"],
                "stage_estimator__classifier__C": [100.0],
                "stage_estimator__classifier__gamma": [0.003, "scale"],
            },
            preprocessing=base,
        ),
    )
    per_level_rows.append(per_level.assign(k=k))
per_level_curve = pd.concat(per_level_rows, ignore_index=True)
split = load_split()


def fit_per_level(k: int) -> Pipeline:
    params = per_level_curve.set_index("k").loc[k, REDUCTION_BEST_PARAMS_COL]
    pipeline = Pipeline(
        [
            ("preprocess", build_preprocessor(base)),
            ("model", make_hierarchical_reduced_svm(k).set_params(**params)),
        ]
    )
    return pipeline.fit(split.X_train, split.y_train)


def level_bands(pipeline: Pipeline) -> dict[str, list[int]]:
    hierarchy = pipeline[-1]
    bands = {"Kultur": hierarchy.crop_estimator_.selected_wavelengths_}
    for crop, stage_model in hierarchy.stage_estimators_.items():
        bands[f"Stadium | {crop}"] = stage_model.selected_wavelengths_
    return bands


# the selection happens inside the model, so the measured bands are the union over all levels
per_level_fits = {k: fit_per_level(k) for k in per_level_curve["k"]}
per_level_curve[N_MEASURED_BANDS_COL] = [
    len({nm for bands in level_bands(per_level_fits[k]).values() for nm in bands})
    for k in per_level_curve["k"]
]
per_level_curve[["k", N_MEASURED_BANDS_COL, *CURVE_METRICS, "bacc_combined_std"]].round(3)

Welche Bänder wählen die Ebenen bei `k=10`? Die Modelle oben wurden auf dem ganzen Trainingsteil gefittet (kein Blick auf die Validierung).

In [ ]:
chosen = level_bands(per_level_fits[10])
pd.DataFrame(
    {"Wellenlängen [nm]": chosen, "Anzahl": {name: len(bands) for name, bands in chosen.items()}}
)

**Befund Bandauswahl je Ebene (Hypothese 2 widerlegt):**

| k je Ebene | gemessene Bänder (Vereinigung) | CV kombiniert |
| --- | --- | --- |
| 5 | 30 | 0,654 |
| 10 | 52 | 0,685 |
| 20 | 88 | 0,738 |
| 40 | 127 | 0,772 |

- **Schlechter als jede globale Variante:** Jede Ebene wählt andere Bänder. Die Vereinigung wächst deshalb schnell (52 Bänder bei k=10), und die Qualität bleibt weit unter Sentinel-2 mit 10 Bändern (0,834) oder jedem 8. Band (0,839).
- **Grund wie in #155:** Die univariate Auswahl greift pro Ebene einen zusammenhängenden Block benachbarter, fast identischer Bänder. Jede Ebene sieht damit nur *eine* Spektralregion.
- **Fachlich sind die gewählten Regionen aber aufschlussreich** (k=10, Trainingsteil):

| Ebene | Gewählte Wellenlängen | Deutung |
| --- | --- | --- |
| Kultur | 600–691 nm | Rot: Chlorophyll-Absorption, Beginn des Red Edge |
| Stadium Mais | 752–864 nm | NIR-Plateau: Biomasse und Blattstruktur wachsen mit dem Stadium |
| Stadium Baumwolle | 783–875 nm, 1094 nm | NIR: Bestandsschluss und Biomasse |
| Stadium Reis | 437–508 nm, 742 nm, 1175 nm | Blau/Grün: bei überflutetem Reisfeld vermutlich das Wasser unter dem Bestand |
| Stadium Soja | 2063–2204 nm, 2335–2345 nm | SWIR2: Trocknung, Lignin und Cellulose bei Reife und Ernte |
| Stadium Winterweizen | 2093–2163 nm, 2274–2285 nm | SWIR2: Abreife und Trocknung des Getreides |

Das ist eine gute Grundlage für die Feature-Importance pro Kultur (#164). Für die Datenreduktion taugt die Auswahl je Ebene nicht.

## 5. Empfehlung nach CV-Regel

Wie in #155: die Variante mit den wenigsten gemessenen Bändern, die höchstens `REDUCTION_MAX_CV_LOSS` (0,01) gegenüber allen Bändern verliert. Die Bandauswahl je Ebene zählt mit der Vereinigung ihrer Bänder.

In [ ]:
candidates = pd.concat(
    [
        curve,
        per_level_curve.assign(
            **{REDUCTION_VARIANT_COL: lambda f: "per_level_" + f["k"].astype(str)}
        ),
    ],
    ignore_index=True,
)
reference = candidates.set_index(REDUCTION_VARIANT_COL).loc["all_bands"]
equivalent = candidates[
    (candidates["bacc_combined"] >= reference["bacc_combined"] - REDUCTION_MAX_CV_LOSS)
    & (candidates[REDUCTION_FAMILY_COL] != "all_bands")
]
recommended = equivalent.nsmallest(1, N_MEASURED_BANDS_COL)
chosen_variants = list(
    dict.fromkeys(["all_bands", *recommended[REDUCTION_VARIANT_COL], "sentinel2"])
)
equivalent[[REDUCTION_VARIANT_COL, N_MEASURED_BANDS_COL, "bacc_combined"]].sort_values(
    N_MEASURED_BANDS_COL
).round(3)

**Empfehlung:** Die Regel wählt **jedes 8. Band** (17 gemessene Bänder, CV 0,839, −0,007). Sentinel-2 verfehlt die Grenze knapp (−0,012) und wird als praktisch verfügbarer Sensor trotzdem mitvalidiert.

## 6. Einmalige Validierung

Die gewählten Varianten werden **einmal** bewertet. Zusätzlich zeigt `oracle_crop_metrics`, wie gut die Stage-Modelle wären, wenn die Kultur immer stimmt. Die Lücke ist das, was Kulturfehler kosten.

In [ ]:
rows = candidates.set_index(REDUCTION_VARIANT_COL)
by_name = {variant.name: variant.preprocessing for variant in variants}
results = {}
oracle = {}
for name in chosen_variants:
    if name.startswith("per_level_"):
        hierarchy = make_hierarchical_reduced_svm(int(name.removeprefix("per_level_")))
        preprocessing = base
    else:
        hierarchy, preprocessing = make_hierarchical_svm(), by_name[name]
    results[name] = run(
        hierarchy.set_params(**rows.loc[name, REDUCTION_BEST_PARAMS_COL]),
        RunConfig(
            name=f"hier_reduction_{name}",
            approach=APPROACH,
            experiment=MLFLOW_REDUCTION_EXPERIMENT,
            description=f"Datenreduktion hierarchisch, Variante {name}, beste CV-Einstellung",
            tuning_run=rows.loc[name, REDUCTION_TUNING_RUN_COL],
            preprocessing=preprocessing,
        ),
    )
    oracle[name] = oracle_crop_metrics(results[name].pipeline, split.X_val, split.y_val)
pd.DataFrame(
    {
        name: {
            **result.metrics.model_dump(),
            "oracle_bacc_stage": oracle[name].bacc_stage,
            "oracle_bacc_combined": oracle[name].bacc_combined,
            **result.input_size._asdict(),
            "run_id": result.run_id[:8],
        }
        for name, result in results.items()
    }
).T

In [ ]:
validation_ax = plot_metric_comparison({name: result.metrics for name, result in results.items()})
validation_ax.set_title("Hierarchisch – Validierung: alle Bänder vs. reduzierte Eingaben")
save_doc_figure(validation_ax.figure, "hierarchisch_reduktion_validierung", MODEL_DOCS_FIGURES_DIR)

## Fazit

| | Gemessene Bänder | CV kombiniert | Val. Kultur | Val. Stadium | Val. kombiniert | Val. kombiniert bei richtiger Kultur |
| --- | --- | --- | --- | --- | --- | --- |
| Alle Bänder (`514156e7`) | 131 | 0,846 | 0,918 | 0,905 | **0,845** | 0,941 |
| Jedes 8. Band (`d8ac6606`) | 17 | 0,839 | 0,920 | 0,910 | 0,828 | 0,924 |
| Sentinel-2 (`8e291c4a`) | 10 | 0,834 | 0,930 | 0,884 | 0,839 | 0,928 |
| *zum Vergleich: kombiniert, alle Bänder (`dac6bb78`)* | 131 | 0,859 | 0,920 | 0,913 | 0,866 | – |

- **Die Datenreduktion gilt auch hierarchisch:** 17 Bänder (jedes 8.) oder 10 Sentinel-2-Bänder verlieren auf der Validierung 0,006–0,017 BAcc kombiniert. Die Aussage aus #155 trägt also für beide Ansätze. Bei Sentinel-2 leidet hierarchisch das Stadium etwas mehr.
- **Die Kultur-Stufe ist der Engpass:** Wäre die Kultur immer richtig, käme der hierarchische Ansatz auf 0,93–0,94 kombiniert. Die Kulturfehler kosten also rund 0,10. Mit Sentinel-2 ist die Kultur sogar etwas besser (0,930 statt 0,918).
- **Bandauswahl je Ebene lohnt sich nicht:** Sie braucht viele Bänder (Vereinigung über alle Ebenen) und bleibt weit hinter den globalen Varianten. Die gewählten Regionen sind aber fachlich gut erklärbar (Rot für die Kultur, NIR für die Stadien von Mais und Baumwolle, SWIR2 für die Reife von Soja und Weizen).
- **Der kombinierte Ansatz bleibt besser:** 0,866 gegenüber 0,845 auf der Validierung, mit allen Bändern und mit reduzierter Eingabe.